# bn_before_vs_after

PDF point (d): batch normalization **before** vs **after** the activation, plus no BN as a control, on the baseline `simple_cnn`.

Workflow: **cell 1** = the configs under comparison, **cell 2** = train them all with the shared `fit()` loop (skips runs that already exist), **cell 3+** = compare. Every run leaves `runs/<member>/<topic>/<run_name>/` with `config.json`, `metrics.json`, `history.csv`, `loss.png`, `metrics.png`, `best.pt`.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while not (ROOT / "pyproject.toml").exists():
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))  # makes `app` importable; `iml` is installed by uv

from iml.config import ExperimentConfig
from iml.data.dataset import DummyDataset, make_loaders
from iml.runs import run_dir
from iml.train import fit
from iml.cli.compare import summary_table, plot_curves

MEMBER, TOPIC = "glhfsox", "bn_before_vs_after"
DUMMY = True  # flip to False once real data loading is wired in load_splits()

## 1. Configurations under comparison

Only the fields that differ between runs are spelled out; everything else is the shared baseline.

In [ ]:
BASE = dict(member=MEMBER, topic=TOPIC, model="simple_cnn", width=16, epochs=5)
CONFIGS = [ExperimentConfig(**BASE, bn_position=pos) for pos in ("before", "after", "none")]
[c.run_name() for c in CONFIGS]

## 2. Train (one run per config, cached on disk)

In [ ]:
def loaders_for(cfg):
    if DUMMY:
        return make_loaders(
            {s: DummyDataset(cfg, n=128, seed=i) for i, s in enumerate(("train", "val", "test"))},
            cfg,
        )
    from iml.cli._data import load_splits

    return load_splits(cfg)


records = []
for cfg in CONFIGS:
    out = run_dir(cfg)
    if (out / "metrics.json").exists():
        from iml.runs import load_run

        print("cached:", out.name)
        records.append(load_run(out))
    else:
        records.append(fit(cfg, loaders_for(cfg)))

## 3. Compare: table, curves, conclusion

In [ ]:
import pandas as pd

pd.set_option("display.width", 200)
table = summary_table(records)
table

In [ ]:
from IPython.display import Image, display

out = run_dir(CONFIGS[0]).parent / "comparison"
out.mkdir(exist_ok=True)
plot_curves(records, out)
table.to_csv(out / "table.csv", index=False)
display(Image(out / "loss.png"), Image(out / "val_metrics.png"))

### Conclusion

_What did we learn? Which variant goes forward? (write it here so the report can quote it)_